# Chapter 3 — Data Ingestion

Data ingestion (also called data gathering) is the process of collecting raw data
from various sources and preparing it for analysis. In this chapter we cover four
key methods: reading **CSV files**, querying a **SQL** database, calling an **API**,
and **web scraping**.

## Working with CSV Files

A CSV (Comma-Separated Values) file stores tabular data as plain text, one record
per line with fields separated by commas. Pandas' `read_csv( )` loads such files
into a DataFrame.

### Loading a CSV File into a DataFrame

In [1]:
# Loading a CSV file with Pandas
# Pandas is a Python library for data manipulation and analysis. Its read_csv( )
# function reads a CSV file into a DataFrame -- a two-dimensional, labeled,
# tabular data structure (rows and columns), much like a spreadsheet.

import pandas as pd

# read_csv( ) accepts a local file path or, as here, a URL pointing directly
# to the raw CSV hosted on GitHub.
url = 'https://raw.githubusercontent.com/kanetkar/LULML/refs/heads/main/ch03/users.csv'

df = pd.read_csv(url)      # load the CSV file into a DataFrame
print(df.head( ))          # display the first five rows for a quick overview


    username firstname lastname   password
0     jsmith      John    Smith   p@ssw3rd
1  janesmith      Jane    Smith     s3cret
2   joesmith       Joe    Smith   blackd0g
3   jimsmith       Jim    Smith  wh1it3cAt
4  jacksmith      Jack    Smith   blu3FisH


### Handling Different Delimiters

Not every file uses commas — some use tabs, semicolons or pipes, and some have no header row at all.

In [2]:
# Handling different delimiters -- first, the problem.
# This file (movie_titles_metadata.tsv) is TAB-separated and has NO header row.
# read_csv( ) assumes a COMMA delimiter by default, so if we do not tell it the
# delimiter or supply column names, things go wrong.
url = 'https://raw.githubusercontent.com/kanetkar/LULML/refs/heads/main/ch03/movie_titles_metadata.tsv'
df = pd.read_csv(url)
print(df.head( ))

# Notice two problems above:
#  1. All six fields are squeezed into a SINGLE column, because no comma was
#     found to split on -- the tab characters (\t) are left inside the values.
#  2. The first movie row is wrongly treated as the column header.


  m0\t10 things i hate about you\t1999\t6.90\t62847\t['comedy' 'romance']
0  m1\t1492: conquest of paradise\t1992\t6.20\t10...                     
1  m2\t15 minutes\t2001\t6.10\t25854\t['action' '...                     
2  m3\t2001: a space odyssey\t1968\t8.40\t163227\...                     
3  m4\t48 hrs.\t1982\t6.90\t22289\t['action' 'com...                     
4  m5\tthe fifth element\t1997\t7.50\t133756\t['a...                     


In [3]:
# Now, the solution -- and its benefit.
# sep = '\t'      tells read_csv( ) that columns are separated by tabs, so each
#                 field lands in its own column instead of one crammed column.
# names = [...]   supplies the column headers, since the file has none -- this
#                 stops the first movie row from being mistaken for the header.
# Together they give us a clean, properly labeled DataFrame ready for analysis.
url = 'https://raw.githubusercontent.com/kanetkar/LULML/refs/heads/main/ch03/movie_titles_metadata.tsv'
df = pd.read_csv(url, sep = '\t',
           names = ['sno','name','release_year','rating','votes','genres'])
print(df.head( ))


  sno                        name release_year  rating     votes  \
0  m0  10 things i hate about you         1999     6.9   62847.0   
1  m1  1492: conquest of paradise         1992     6.2   10421.0   
2  m2                  15 minutes         2001     6.1   25854.0   
3  m3       2001: a space odyssey         1968     8.4  163227.0   
4  m4                     48 hrs.         1982     6.9   22289.0   

                                           genres  
0                            ['comedy' 'romance']  
1     ['adventure' 'biography' 'drama' 'history']  
2           ['action' 'crime' 'drama' 'thriller']  
3                ['adventure' 'mystery' 'sci-fi']  
4  ['action' 'comedy' 'crime' 'drama' 'thriller']  


### Handling a Misplaced Header

Sometimes the real column names are not on the first row. The `header` parameter tells Pandas which row to use as the header.

In [4]:
# Handling a misplaced header -- first, the problem.
# By default read_csv( ) treats the FIRST row (row 0) as the column headers.
# But in test.csv the real headers sit on the SECOND line; row 0 is a stray row.
url = 'https://raw.githubusercontent.com/kanetkar/LULML/refs/heads/main/ch03/test.csv'
df = pd.read_csv(url)
print(df.head( ))

# Notice the problem above:
#  - Pandas labels the columns 'Unnamed: 0', 'Unnamed: 1', ... because row 0
#    held no proper names.
#  - The actual headers (enrollee_id, city, ...) get pushed down and are
#    treated as ordinary data instead of column names.


   Unnamed: 0   Unnamed: 1 Unnamed: 2              Unnamed: 3 Unnamed: 4  \
0           0  enrollee_id       city  city_development_index     gender   
1           1        29725    city_40                   0.776       Male   
2           2        11561    city_21                   0.624        NaN   
3           3        33241   city_115                   0.789        NaN   
4           4          666   city_162                   0.767       Male   

                Unnamed: 5           Unnamed: 6       Unnamed: 7  \
0      relevent_experience  enrolled_university  education_level   
1   No relevent experience        no_enrollment         Graduate   
2   No relevent experience     Full time course         Graduate   
3   No relevent experience                  NaN         Graduate   
4  Has relevent experience        no_enrollment          Masters   

         Unnamed: 8  Unnamed: 9   Unnamed: 10     Unnamed: 11   Unnamed: 12  \
0  major_discipline  experience  company_size    compan

In [5]:
# Now, the solution -- and its benefit.
# header = 1 tells read_csv( ) that the column names live on line 1 (the second
# row, since counting starts at 0). Pandas then uses that row as the headers and
# discards the stray row 0 above it.
# The benefit: a DataFrame with correct, meaningful column names ready for use.
url = 'https://raw.githubusercontent.com/kanetkar/LULML/refs/heads/main/ch03/test.csv'
df = pd.read_csv(url, header = 1)
print(df.head( ))


   0  enrollee_id      city  city_development_index gender  \
0  1        29725   city_40                   0.776   Male   
1  2        11561   city_21                   0.624    NaN   
2  3        33241  city_115                   0.789    NaN   
3  4          666  city_162                   0.767   Male   

       relevent_experience enrolled_university education_level  \
0   No relevent experience       no_enrollment        Graduate   
1   No relevent experience    Full time course        Graduate   
2   No relevent experience                 NaN        Graduate   
3  Has relevent experience       no_enrollment         Masters   

  major_discipline experience company_size    company_type last_new_job  \
0             STEM         15        50-99         Pvt Ltd           >4   
1             STEM          5          NaN             NaN        never   
2  Business Degree         <1          NaN         Pvt Ltd        never   
3             STEM        >20        50-99  Funded Startup

### Handling Encoding Issues

Files with non-English characters may not be valid UTF-8. Specifying the correct `encoding` avoids decoding errors.

In [6]:
# Handling encoding issues -- first, the problem.
# zomato.csv contains special characters that are NOT valid UTF-8. By default
# read_csv( ) decodes files as UTF-8, so loading it without an encoding fails.
url = 'https://raw.githubusercontent.com/kanetkar/LULML/refs/heads/main/ch03/zomato.csv'
try :
    df = pd.read_csv(url)               # no encoding given -> defaults to UTF-8
    print(df.head( ))
except UnicodeDecodeError as e :
    print("UnicodeDecodeError:", e)

# The error above shows Pandas could not decode a byte as UTF-8 -- the classic
# symptom of an encoding mismatch.


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xed in position 7044: invalid continuation byte


In [7]:
# Now, the solution -- and its benefit.
# encoding = 'latin-1' tells read_csv( ) to read the file using Latin-1, which
# handles the special (often European) characters that broke UTF-8 decoding.
# The benefit: the file loads cleanly into a DataFrame instead of raising an error.
url = 'https://raw.githubusercontent.com/kanetkar/LULML/refs/heads/main/ch03/zomato.csv'
df = pd.read_csv(url, encoding = 'latin-1')
print(df.head( ))


   Restaurant ID         Restaurant Name  Country Code              City  \
0        6317637        Le Petit Souffle           162       Makati City   
1        6304287        Izakaya Kikufuji           162       Makati City   
2        6300002  Heat - Edsa Shangri-La           162  Mandaluyong City   
3        6318506                    Ooma           162  Mandaluyong City   
4        6314302             Sambo Kojin           162  Mandaluyong City   

                                             Address  \
0  Third Floor, Century City Mall, Kalayaan Avenu...   
1  Little Tokyo, 2277 Chino Roces Avenue, Legaspi...   
2  Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...   
3  Third Floor, Mega Fashion Hall, SM Megamall, O...   
4  Third Floor, Mega Atrium, SM Megamall, Ortigas...   

                                     Locality  \
0   Century City Mall, Poblacion, Makati City   
1  Little Tokyo, Legaspi Village, Makati City   
2  Edsa Shangri-La, Ortigas, Mandaluyong City   
3      SM 

### Handling Large Datasets in Chunks

Very large files may not fit in memory. The `chunksize` parameter lets us read and process the file a few rows at a time.

In [8]:
# Reading large files in chunks -- first, the problem.
# pd.read_csv( ) loads the ENTIRE file into a single DataFrame held in memory.
# That is fine for this file, but for a dataset with millions of rows or several
# gigabytes it can exhaust RAM and raise a memory error.
url = 'https://raw.githubusercontent.com/kanetkar/LULML/refs/heads/main/ch03/aug_train.csv'
df = pd.read_csv(url)                        # whole file loaded at once
print('rows loaded at once:', len(df))
print('memory used (MB): %.1f' % (df.memory_usage(deep = True).sum() / 1e6))

# Every row sits in memory simultaneously -- this approach does not scale to
# very large files.


rows loaded at once: 19158
memory used (MB): 12.4


In [9]:
# Now, the solution -- and its benefit.
# chunksize = 1000 makes read_csv( ) return an iterator that yields the file
# 1000 rows at a time, so only a small slice is in memory at any one moment.
url = 'https://raw.githubusercontent.com/kanetkar/LULML/refs/heads/main/ch03/aug_train.csv'
chunk_size = 1000
chunks = pd.read_csv(url, chunksize = chunk_size)

for i, chunk in enumerate(chunks, start = 1) :
      print('chunk', i, '-> rows:', len(chunk))   # process each chunk here

# The benefit: we can process a dataset far larger than memory by handling it
# one manageable chunk at a time.


chunk 1 -> rows: 1000
chunk 2 -> rows: 1000
chunk 3 -> rows: 1000
chunk 4 -> rows: 1000
chunk 5 -> rows: 1000
chunk 6 -> rows: 1000
chunk 7 -> rows: 1000
chunk 8 -> rows: 1000
chunk 9 -> rows: 1000
chunk 10 -> rows: 1000
chunk 11 -> rows: 1000
chunk 12 -> rows: 1000
chunk 13 -> rows: 1000
chunk 14 -> rows: 1000
chunk 15 -> rows: 1000
chunk 16 -> rows: 1000
chunk 17 -> rows: 1000
chunk 18 -> rows: 1000
chunk 19 -> rows: 1000
chunk 20 -> rows: 158


## Data Ingestion Using SQL

Much business-critical data lives in SQL databases. We use SQLite (built into
Python via the `sqlite3` module) to create a small `world.db`, then query it with
Pandas.

In [10]:
# Data Ingestion using SQL -- creating and populating the database.
# SQLite is a lightweight SQL database that ships built-in with Python (via the
# sqlite3 module), so no separate server is needed. This cell connects to
# 'world.db' (creating the file if it does not exist), then uses a cursor to
# run SQL: CREATE TABLE builds the city, country and language tables, INSERT
# adds sample rows, and conn.commit( ) saves the changes to disk.

import sqlite3
conn = sqlite3.connect('world.db', timeout = 10)
cursor = conn.cursor( )

# start fresh so this cell can be re-run safely -- remove any existing tables
cursor.execute("DROP TABLE IF EXISTS city")
cursor.execute("DROP TABLE IF EXISTS country")
cursor.execute("DROP TABLE IF EXISTS language")

# create the city table
qry = '''CREATE TABLE IF NOT EXISTS city (
        ID INTEGER PRIMARY KEY AUTOINCREMENT,
        Name TEXT NOT NULL,
        CountryCode TEXT NOT NULL,
        District TEXT NOT NULL,
        Population INTEGER NOT NULL )'''
cursor.execute(qry)

# create the country table
qry = '''CREATE TABLE IF NOT EXISTS country (
        Code TEXT PRIMARY KEY, Name TEXT NOT NULL,
        Continent TEXT NOT NULL, Population INTEGER NOT NULL,
        LifeExpectancy REAL )'''
cursor.execute(qry)

# create the language table
qry = '''CREATE TABLE IF NOT EXISTS language (
        ID INTEGER PRIMARY KEY AUTOINCREMENT,
        CountryCode TEXT NOT NULL, Language TEXT NOT NULL,
        IsOfficial TEXT NOT NULL, Percentage REAL NOT NULL,
        FOREIGN KEY (CountryCode) REFERENCES country (Code))'''
cursor.execute(qry)

# Insert data into city table
cursor.execute ( "INSERT INTO city (Name, CountryCode, District, Population ) VALUES ( 'Kabul', 'AFG', 'Kabol', 1780000 )" )
cursor.execute ( "INSERT INTO city (Name, CountryCode, District, Population ) VALUES ( 'New York', 'USA', 'New York', 8175133 )" )
cursor.execute ( "INSERT INTO city (Name, CountryCode, District, Population) VALUES ( 'Tokyo', 'JPN', 'Tokyo', 9273000 )" )

# Insert data into country table
cursor.execute ( "INSERT INTO country (Code, Name, Continent, Population, LifeExpectancy ) VALUES ( 'AFG', 'Afghanistan', 'Asia', 38928341, 64.8 )" )
cursor.execute ( "INSERT INTO country ( Code, Name, Continent, Population, LifeExpectancy) VALUES ( 'USA', 'United States', 'North America', 331002651, 78.9 )" )
cursor.execute ( "INSERT INTO country (Code, Name, Continent, Population, LifeExpectancy ) VALUES ('JPN', 'Japan', 'Asia', 126476461, 84.5 )" )

# Insert data into language table
cursor.execute ( "INSERT INTO language (CountryCode, Language, IsOfficial, Percentage ) VALUES ( 'AFG', 'Pashto', 'T', 52.4 )" )
cursor.execute ( "INSERT INTO language (CountryCode, Language, IsOfficial, Percentage ) VALUES ( 'USA', 'English', 'T', 79.0 )" )
cursor.execute ( "INSERT INTO language (CountryCode, Language, IsOfficial, Percentage ) VALUES ( 'JPN', 'Japanese', 'T', 99.2 )" )

# Commit the changes to the database
conn.commit( )

# close the connection so it does not keep the database locked
conn.close( )


In [11]:
# Querying the database -- loading SQL results into a DataFrame.
# pd.read_sql_query( ) runs a SQL query against a database connection and returns
# the result directly as a Pandas DataFrame. It takes two arguments: the SQL
# query string and the connection object. Here we fetch every row of the city
# table, then close the connection.

import sqlite3
conn = sqlite3.connect('world.db', timeout = 10)
city_df = pd.read_sql_query("SELECT * FROM city", conn)
print(city_df)
conn.close( )


   ID      Name CountryCode  District  Population
0   1     Kabul         AFG     Kabol     1780000
1   2  New York         USA  New York     8175133
2   3     Tokyo         JPN     Tokyo     9273000


In [12]:
# Filtering rows with a WHERE clause.
# To pull only the data relevant to our analysis, we add a SQL WHERE clause to
# the query -- here, only cities with a population greater than 5 million. The
# filtered result is returned as a Pandas DataFrame just like before.

conn = sqlite3.connect('world.db', timeout = 10)
# retrieve only cities with a population greater than 5 million
qry = "SELECT * FROM city WHERE Population > 5000000"
city_df = pd.read_sql_query(qry, conn)
print(city_df)
conn.close( )


   ID      Name CountryCode  District  Population
0   2  New York         USA  New York     8175133
1   3     Tokyo         JPN     Tokyo     9273000


## Data Ingestion Using an API

An API (Application Programming Interface) lets us fetch data from an external
service over the web. Here we call the free JSONPlaceholder API and load the JSON
response into a DataFrame.

In [13]:
# Data Ingestion using an API.
# An API (Application Programming Interface) lets us fetch data from an external
# service over the web. Here we use the free JSONPlaceholder API, which returns
# user records as JSON. We send an HTTP GET request with the requests library,
# check that it succeeded (status code 200), parse the JSON response, and load
# it straight into a Pandas DataFrame.

import requests

# API endpoint
url = "https://jsonplaceholder.typicode.com/users"

# send a GET request to the API
response = requests.get(url)

# check if the request was successful
if response.status_code == 200 :
    data = response.json( )  # parse the response as JSON
    users_df = pd.DataFrame(data)    # convert JSON data to  DataFrame
    print(users_df.head( ))   # display first few rows of the DataFrame
else :
    print("Error: Unable to fetch data")


   id              name   username                      email  \
0   1     Leanne Graham       Bret          Sincere@april.biz   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca   

                                             address                  phone  \
0  {'street': 'Kulas Light', 'suite': 'Apt. 556',...  1-770-736-8031 x56442   
1  {'street': 'Victor Plains', 'suite': 'Suite 87...    010-692-6593 x09125   
2  {'street': 'Douglas Extension', 'suite': 'Suit...         1-463-123-4447   
3  {'street': 'Hoeger Mall', 'suite': 'Apt. 692',...      493-170-9623 x156   
4  {'street': 'Skiles Walks', 'suite': 'Suite 351...          (254)954-1289   

         website                                            company  
0  hildegard.org  {'name': 'Romaguera-Crona', 'catchPhrase': 'Mu

## Data Ingestion Using Web Scraping

When a site offers no API, we can extract data directly from its HTML using
`requests` and BeautifulSoup. Always respect a site's `robots.txt` and terms of
service before scraping.

In [14]:
# Data Ingestion using Web Scraping.
# When a website offers no API, we can extract data directly from its HTML. Here
# we fetch a page from books.toscrape.com, parse the HTML with BeautifulSoup,
# and pull out each book's title, price, image URL and availability. The values
# are collected into lists and finally assembled into a Pandas DataFrame.
# (Always check a site's robots.txt and terms of service before scraping.)

import requests
from bs4 import BeautifulSoup
import pandas as pd

# the web page we want to scrape
url = "https://books.toscrape.com/catalogue/page-1.html"

# send an HTTP GET request to download the page's HTML
response = requests.get(url)

# stop early if the page could not be fetched (200 means success)
if response.status_code != 200 :
      print("Failed to retrieve the webpage.")
      exit( )

# parse the raw HTML text into a navigable BeautifulSoup object
soup = BeautifulSoup(response.text, 'html.parser')

# every book on the page sits inside an <article class="product_pod"> tag;
# find_all( ) returns a list of all such book containers
books = soup.find_all('article', class_='product_pod')

# empty lists to collect one field per book
titles, prices, images, availabilities = [ ], [ ], [ ], [ ]

# walk through each book container and pull out the fields we need
for book in books :
      # title is stored in the 'title' attribute of the <a> inside the <h3>
      title = book.h3.a['title']

      # price text lives in a <p class="price_color">
      price = book.find('p', class_='price_color').text

      # the <img> src is a relative path, so prepend the base URL to make it absolute
      image = "https://books.toscrape.com/" + book.find('img')['src'].replace('../', '')

      # availability text is in a <p class="instock availability">; strip( ) trims whitespace
      availability = book.find('p', class_='instock availability').text.strip( )

      # add this book's values to their respective lists
      titles.append(title)
      prices.append(price)
      images.append(image)
      availabilities.append(availability)

# combine the four parallel lists into a single DataFrame, one column each
books_df = pd.DataFrame({
      'Title': titles,
      'Price': prices,
      'Image URL': images,
      'Availability': availabilities })
print(books_df)


                                                Title    Price  \
0                                A Light in the Attic  Â£51.77   
1                                  Tipping the Velvet  Â£53.74   
2                                          Soumission  Â£50.10   
3                                       Sharp Objects  Â£47.82   
4               Sapiens: A Brief History of Humankind  Â£54.23   
5                                     The Requiem Red  Â£22.65   
6   The Dirty Little Secrets of Getting Your Dream...  Â£33.34   
7   The Coming Woman: A Novel Based on the Life of...  Â£17.93   
8   The Boys in the Boat: Nine Americans and Their...  Â£22.60   
9                                     The Black Maria  Â£52.15   
10     Starving Hearts (Triangular Trade Trilogy, #1)  Â£13.99   
11                              Shakespeare's Sonnets  Â£20.66   
12                                        Set Me Free  Â£17.46   
13  Scott Pilgrim's Precious Little Life (Scott Pi...  Â£52.29   
14        